In [1]:
# Cell 1 – Setup
!pip install duckdb pandas datasets transformers peft accelerate

import duckdb
import pandas as pd
import textwrap


In [7]:

# 1. This is the converted DIRECT download link for your file ID
csv_path = "https://drive.google.com/file/d/1nON5KbGGszgy5wsPoVtLVGcI5sbh51pq/view?usp=drive_link"

con = duckdb.connect("transactions.duckdb")

# 2. IMPORTANT: We must DROP the table first because it currently contains HTML junk
con.execute("DROP TABLE IF EXISTS transactions")

# 3. Load from the direct link
con.execute(f"""
    CREATE TABLE transactions AS
    SELECT *
    FROM read_csv_auto('{csv_path}', HEADER=TRUE)
""")

# 4. Verification
print("Total Rows:", con.execute("SELECT COUNT(*) FROM transactions").fetchall()[0][0])
print("\nColumn names found:")
print(con.execute("PRAGMA table_info('transactions')").fetchdf()[['name', 'type']])

Total Rows: 0

Column names found:
                                                 name     type
0   </script><meta property="og:title" content="Cr...  VARCHAR
1   usp=embed_facebook"><link rel="shortcut icon" ...  VARCHAR
2   </script><link rel="stylesheet" href="https://...  VARCHAR
3                                                 500  VARCHAR
4   700" rel="stylesheet" nonce="eb8XnzOmVO8DTZJgt...  VARCHAR
..                                                ...      ...
90        linkEl.onload = ()=> {linkEl.media = 'all'}  VARCHAR
91                       linkEl.removeAttribute('id')  VARCHAR
92  </script><script nonce="MDVNtaWqtuShJxW9DzTc0Q...  VARCHAR
93  </script><script id="base-js" async type="text...  VARCHAR
94  </script></body></html>                       ...  VARCHAR

[95 rows x 2 columns]


In [12]:
# 1. Use gdown to download the actual CSV to the Colab local disk
# This tool handles the Google Drive 'Large File' warning automatically.
file_id = "1nON5KbGGszgy5wsPoVtLVGcI5sbh51pq"
local_csv = "credit_card_data.csv"

if not os.path.exists(local_csv):
    print("Downloading actual CSV file... this may take a moment.")
    !gdown --id {file_id} -O {local_csv}

# 2. Connect and RESET the table
con = duckdb.connect("transactions.duckdb")
con.execute("DROP TABLE IF EXISTS transactions")

# 3. Load from the LOCAL file (not the URL)
print("Loading data into DuckDB...")
con.execute(f"CREATE TABLE transactions AS SELECT * FROM read_csv_auto('{local_csv}')")

# 4. PERFORM THE INSPECTION
print("\n--- VALUE RANGES FOR REALISTIC QUESTIONS ---")
df_ranges = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        MIN(amt) as min_amt,
        MAX(amt) as max_amt,
        MIN(trans_date_trans_time) as first_trans,
        MAX(trans_date_trans_time) as last_trans,
        COUNT(DISTINCT cc_num) as unique_cards,
        COUNT(DISTINCT category) as unique_categories
    FROM transactions
""").fetchdf()

pd.set_option('display.max_columns', None)
print(df_ranges)

# 5. Get your sample for the next steps
df_sample = con.execute("""
    SELECT cc_num, MIN(trans_date_trans_time) AS min_dt, MAX(trans_date_trans_time) AS max_dt
    FROM transactions
    GROUP BY cc_num
    LIMIT 1000
""").fetchdf()

print("\n--- SAMPLE FOR GENERATING QUESTIONS ---")
print(df_sample.head())

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1nON5KbGGszgy5wsPoVtLVGcI5sbh51pq
From (redirected): https://drive.google.com/uc?id=1nON5KbGGszgy5wsPoVtLVGcI5sbh51pq&confirm=t&uuid=c246e9d5-a5db-4e55-90f0-5d3bf08fb036
To: /content/credit_card_data.csv
100% 354M/354M [00:03<00:00, 114MB/s]
Loading data into DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- VALUE RANGES FOR REALISTIC QUESTIONS ---
   total_rows  min_amt  max_amt         first_trans          last_trans  \
0     1296675      1.0  28948.9 2019-01-01 00:00:18 2020-06-21 12:13:37   

   unique_cards  unique_categories  
0           983                 14  

--- SAMPLE FOR GENERATING QUESTIONS ---
                cc_num              min_dt              max_dt
0       36078114201167 2019-01-01 02:38:28 2020-06-21 01:24:18
1         630469040731 2019-01-01 10:29:29 2020-06-20 19:35:19
2     6011381817520024 2019-01-01 06:52:31 2020-06-21 08:19:57
3  4208110975550360171 2019-01-01 15:07:15 2020-06-21 03:23:01
4  4457732997086323466 2019-01-01 04:14:13 2020-06-21 06:44:03


In [11]:
import os

In [13]:
# Cell 4 – Generate (question, sql) pairs

import random
from datetime import datetime

def random_date_range(min_dt, max_dt):
    # DuckDB will probably give strings; adjust if they’re timestamps
    min_ts = pd.to_datetime(min_dt)
    max_ts = pd.to_datetime(max_dt)
    if min_ts >= max_ts:
        return min_ts, max_ts
    # choose 2 random dates in range
    t1 = min_ts + (max_ts - min_ts) * random.random()
    t2 = min_ts + (max_ts - min_ts) * random.random()
    start, end = sorted([t1, t2])
    return start.date().isoformat(), end.date().isoformat()

def q_total_spent_by_card_and_range(row):
    card = int(row["cc_num"])
    start, end = random_date_range(row["min_dt"], row["max_dt"])
    question = (
        f"What is the total amount spent by card {card} "
        f"between {start} and {end}?"
    )
    sql = textwrap.dedent(f"""
        SELECT SUM(amt) AS total_amount
        FROM transactions
        WHERE cc_num = {card}
          AND trans_date_trans_time >= '{start}'
          AND trans_date_trans_time < '{end}';
    """).strip()
    return question, sql

def q_top_merchants_for_card(card):
    question = f"Which merchants has card {card} spent the most money at?"
    sql = textwrap.dedent(f"""
        SELECT merchant, SUM(amt) AS total_spent
        FROM transactions
        WHERE cc_num = {card}
        GROUP BY merchant
        ORDER BY total_spent DESC
        LIMIT 10;
    """).strip()
    return question, sql

def q_fraud_rate_by_category():
    question = "What is the fraud rate by transaction category?"
    sql = textwrap.dedent("""
        SELECT category,
               AVG(is_fraud::DOUBLE) AS fraud_rate,
               COUNT(*) AS total_txn
        FROM transactions
        GROUP BY category
        ORDER BY fraud_rate DESC;
    """).strip()
    return question, sql


In [14]:
# Cell 5 – Build training records

records = []

for _, row in df_sample.iterrows():
    q, s = q_total_spent_by_card_and_range(row)
    records.append({"question": q, "sql": s, "type": "card_total"})

    q, s = q_top_merchants_for_card(int(row["cc_num"]))
    records.append({"question": q, "sql": s, "type": "top_merchants"})

# Add some global-style questions
for _ in range(300):
    q, s = q_fraud_rate_by_category()
    records.append({"question": q, "sql": s, "type": "fraud_rate_category"})

len(records), records[0]


(2266,
 {'question': 'What is the total amount spent by card 36078114201167 between 2019-05-07 and 2019-12-05?',
  'sql': "SELECT SUM(amt) AS total_amount\nFROM transactions\nWHERE cc_num = 36078114201167\n  AND trans_date_trans_time >= '2019-05-07'\n  AND trans_date_trans_time < '2019-12-05';",
  'type': 'card_total'})

In [15]:
# Cell 6 – Convert to a text field for training

from datasets import Dataset

def format_example(ex):
    # You can tweak this prompt later
    return {
        "text": textwrap.dedent(f"""
        ### Instruction:
        You are an assistant that writes SQL queries for DuckDB.
        Given the schema below and the user's question, write a valid SQL query.

        ### Schema:
        Table transactions(
            row_index INT,
            trans_date_trans_time TIMESTAMP,
            cc_num BIGINT,
            merchant TEXT,
            category TEXT,
            amt DOUBLE,
            first TEXT,
            last TEXT,
            gender TEXT,
            street TEXT,
            city TEXT,
            state TEXT,
            zip INT,
            lat DOUBLE,
            long DOUBLE,
            city_pop INT,
            job TEXT,
            dob DATE,
            trans_num TEXT,
            unix_time BIGINT,
            merch_lat DOUBLE,
            merch_long DOUBLE,
            is_fraud INT,
            merch_zipcode INT
        )

        ### Question:
        {ex['question']}

        ### Response (SQL only):
        {ex['sql']}
        """).strip()
    }

ds = Dataset.from_list(records)
ds = ds.map(format_example)
ds = ds.remove_columns([c for c in ds.column_names if c != "text"])
ds_train = ds.shuffle(seed=42)
ds_train


Map:   0%|          | 0/2266 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2266
})

In [17]:
# Cell 7 – Load model + tokenizer with 4-bit quantization
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- FIX START: Use BitsAndBytesConfig ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Pass config here instead of load_in_4bit
    device_map="auto",
    trust_remote_code=True
)
# --- FIX END ---

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


In [18]:
# Cell 8 – Tokenize dataset

max_length = 1024  # queries+SQL are short; this is fine

def tokenize_function(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )
    # For causal language modeling, labels are usually the input_ids
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

tokenized_ds = ds_train.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_ds = tokenized_ds.train_test_split(test_size=0.05, seed=42)
tokenized_ds

Map:   0%|          | 0/2266 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2152
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 114
    })
})

In [19]:
# Cell 9 – TrainingArguments + Trainer

output_dir = "nl2sql-cc-finetuned"

# Upgrade accelerate to ensure compatibility with the installed transformers version.
# An old accelerate version can cause issues with TrainingArguments.
!pip install --upgrade accelerate

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=0.5,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    fp16=True,
    report_to="none",
    lr_scheduler_type="cosine",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,2.646316


('nl2sql-cc-finetuned/tokenizer_config.json',
 'nl2sql-cc-finetuned/chat_template.jinja',
 'nl2sql-cc-finetuned/tokenizer.json')

In [23]:
# Cell 10 – Load fine-tuned model for inference
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Configuration - ensure these match your training settings
# From your previous error, your folder is named 'nl2sql-cc-finetuned'
model_name = "Qwen/Qwen2.5-3B-Instruct"
ft_model_name = "./nl2sql-cc-finetuned"

# 2. BitsAndBytes Config (Fixes the TypeError)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 3. Load the base model
print(f"Loading base model: {model_name}...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 4. Load the PEFT adapter
# We check if the folder exists first to avoid path errors
if os.path.exists(ft_model_name):
    print(f"Loading adapter from {ft_model_name}...")
    model = PeftModel.from_pretrained(base_model, ft_model_name)

    # IMPORTANT: We DO NOT merge_and_unload() here.
    # Quantized 4-bit models cannot be merged.
    # The PeftModel wrapper handles inference automatically.

    # Load tokenizer from the fine-tuned folder
    tokenizer = AutoTokenizer.from_pretrained(ft_model_name)
else:
    raise FileNotFoundError(f"Could not find the folder: {ft_model_name}. Please check your sidebar.")

# 5. Prepare for inference
model.eval()
print("Model and adapter loaded successfully and ready for inference!")

Loading base model: Qwen/Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading adapter from ./nl2sql-cc-finetuned...
Model and adapter loaded successfully and ready for inference!


In [24]:
# Cell 11 – NL -> SQL generation

SCHEMA_TEXT = """
Table transactions(
    row_index INT,
    trans_date_trans_time TIMESTAMP,
    cc_num BIGINT,
    merchant TEXT,
    category TEXT,
    amt DOUBLE,
    first TEXT,
    last TEXT,
    gender TEXT,
    street TEXT,
    city TEXT,
    state TEXT,
    zip INT,
    lat DOUBLE,
    long DOUBLE,
    city_pop INT,
    job TEXT,
    dob DATE,
    trans_num TEXT,
    unix_time BIGINT,
    merch_lat DOUBLE,
    merch_long DOUBLE,
    is_fraud INT,
    merch_zipcode INT
)
"""

def build_prompt(question: str) -> str:
    return textwrap.dedent(f"""
    ### Instruction:
    You are an assistant that writes SQL queries for DuckDB.
    Only output a single SQL query, nothing else.

    ### Schema:
    {SCHEMA_TEXT}

    ### Question:
    {question}

    ### Response (SQL only):
    """).strip()

def generate_sql(question: str, max_new_tokens: int = 256) -> str:
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Extract everything after the last marker
    if "### Response (SQL only):" in full_text:
        sql_part = full_text.split("### Response (SQL only):")[-1].strip()
    else:
        sql_part = full_text.strip()
    # Optional: strip backticks and trailing commentary
    sql_part = sql_part.split(";")[0].strip() + ";"
    return sql_part


In [25]:
# Cell 12 – Execute generated SQL on DuckDB

def query_agent(question: str):
    sql = generate_sql(question)
    print("Generated SQL:\n", sql)

    try:
        df = con.execute(sql).fetchdf()
    except Exception as e:
        print("SQL execution error:", e)
        df = pd.DataFrame()

    return sql, df

# Example:
q = "Show the top 5 merchants by total fraud amount."
sql, df = query_agent(q)
df.head()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated SQL:
 ;
SQL execution error: 'NoneType' object has no attribute 'fetchdf'


""


In [26]:
# Optional – simple Gradio app

!pip install gradio

import gradio as gr

def gradio_interface(question):
    sql, df = query_agent(question)
    return sql, df

demo = gr.Interface(
    fn=gradio_interface,
    inputs=gr.Textbox(label="Ask about your transactions"),
    outputs=[
        gr.Textbox(label="Generated SQL"),
        gr.Dataframe(label="Result")
    ],
    title="Credit Card Transactions Query Assistant"
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://db27354b3bbad0df24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
# Cell 13 – (run once) Install Gradio + Matplotlib
!pip install gradio matplotlib --quiet


In [28]:
# Cell 14 – Extra imports for viz + Gradio

import matplotlib.pyplot as plt
import gradio as gr
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype


In [29]:
# Cell 15 – Heuristics to choose chart type and build a matplotlib figure

def choose_axes(df: pd.DataFrame, question: str):
    """
    Decide which columns to plot (x, y) based on NL question + df.
    Returns (x_col, y_col, chart_type) or (None, None, None) if can't decide.

    chart_type in {"line", "bar", "scatter"}.
    """
    if df.empty or df.shape[1] == 0:
        return None, None, None

    cols = list(df.columns)

    # Look for time-related phrasing in the question
    q_lower = question.lower()
    time_keywords = [
        "over time", "by day", "by month", "by year",
        "trend", "time series", "daily", "monthly"
    ]
    is_time_question = any(k in q_lower for k in time_keywords)

    # Identify numeric, datetime, and "categorical" columns
    numeric_cols = [c for c in cols if is_numeric_dtype(df[c])]
    datetime_cols = [c for c in cols if is_datetime64_any_dtype(df[c])]
    cat_cols = [
        c for c in cols
        if df[c].dtype == "object" or str(df[c].dtype).startswith("category")
    ]

    # If user is asking for a trend but we don't have datetime dtype,
    # try to parse datetime-like columns.
    if is_time_question and not datetime_cols:
        for c in cols:
            if "date" in c.lower() or "time" in c.lower():
                try:
                    df[c] = pd.to_datetime(df[c])
                    datetime_cols.append(c)
                except Exception:
                    pass

    # 1) Time series: datetime x numeric y → line chart
    if is_time_question and datetime_cols and numeric_cols:
        x_col = datetime_cols[0]
        candidate_numeric = [c for c in numeric_cols if "id" not in c.lower()]
        y_col = candidate_numeric[0] if candidate_numeric else numeric_cols[0]
        return x_col, y_col, "line"

    # 2) Categorical vs numeric → bar chart
    if cat_cols and numeric_cols:
        x_col = cat_cols[0]
        candidate_numeric = [c for c in numeric_cols if c != x_col]
        y_col = candidate_numeric[0] if candidate_numeric else numeric_cols[0]
        return x_col, y_col, "bar"

    # 3) Two numeric columns → scatter plot
    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[1]
        return x_col, y_col, "scatter"

    # fallback: no clear visualization
    return None, None, None


def create_plot(df: pd.DataFrame, question: str):
    """
    Create a matplotlib figure based on df + natural-language question.
    Returns a figure or None if plotting doesn't make sense.
    """
    if df is None or df.empty:
        return None

    x_col, y_col, chart_type = choose_axes(df, question)
    if x_col is None or y_col is None or chart_type is None:
        return None

    # Limit rows to keep plots light-weight
    df_plot = df[[x_col, y_col]].dropna().copy()
    if df_plot.empty:
        return None

    max_points = 500
    if len(df_plot) > max_points:
        df_plot = df_plot.sample(max_points, random_state=42)

    fig, ax = plt.subplots()

    if chart_type == "line":
        df_plot = df_plot.sort_values(by=x_col)
        ax.plot(df_plot[x_col], df_plot[y_col])
    elif chart_type == "bar":
        # Aggregate categories if there are many of them
        if df_plot[x_col].nunique() > 20:
            df_agg = df_plot.groupby(x_col, as_index=False)[y_col].sum()
            df_agg = df_agg.sort_values(by=y_col, ascending=False).head(20)
        else:
            df_agg = df_plot
        ax.bar(df_agg[x_col], df_agg[y_col])
        ax.set_xticklabels(df_agg[x_col], rotation=45, ha="right")
    elif chart_type == "scatter":
        ax.scatter(df_plot[x_col], df_plot[y_col])

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(question)
    fig.tight_layout()
    return fig


In [30]:
# Cell 16 – Extended agent that includes visualization

def query_agent_with_viz(question: str):
    """
    Natural language → SQL → DuckDB → (DataFrame, matplotlib Figure).

    Returns: sql, df, fig, error_message
    """
    if not question.strip():
        return "", pd.DataFrame(), None, "Please enter a question."

    # NL -> SQL
    try:
        sql = generate_sql(question)
    except Exception as e:
        return "", pd.DataFrame(), None, f"Error generating/cleaning SQL: {e}"

    # Execute SQL on DuckDB
    try:
        df = con.execute(sql).fetchdf()
    except Exception as e:
        return sql, pd.DataFrame(), None, f"SQL execution error: {e}"

    # Build visualization (if possible)
    try:
        fig = create_plot(df, question)
    except Exception as e:
        fig = None
        err_msg = f"Query ran, but failed to create visualization: {e}"
        return sql, df, fig, err_msg

    # No errors
    return sql, df, fig, ""


In [31]:
# Quick smoke test (optional)
sql, df, fig, err = query_agent_with_viz("Plot the fraud rate by category.")
print("SQL:", sql)
print("Error:", err)
df.head()


SQL: ;
Error: SQL execution error: 'NoneType' object has no attribute 'fetchdf'


""


In [32]:
# Cell 17 – Gradio UI: NL → SQL → Table + Visualization

example_questions = [
    "Show the top 10 merchants by total transaction amount and plot them.",
    "Plot the fraud rate by transaction category.",
    "Show daily total fraud amount over time.",
    "Compare average transaction amount by state and visualize it.",
    "Plot the relationship between transaction amount and city population."
]

with gr.Blocks(title="Credit Card Transactions NL→SQL + Visualization") as demo:
    gr.Markdown("# 💳 Credit Card Transactions Query & Visualization Assistant")
    gr.Markdown(
        "Ask questions in plain English. The assistant will:\n"
        "1. Generate SQL for DuckDB,\n"
        "2. Run it on your 1.3M-row dataset,\n"
        "3. Show the table result and an automatic chart when possible."
    )

    with gr.Row():
        question_box = gr.Textbox(
            label="Your question",
            placeholder="e.g., Plot the fraud rate by category.",
            lines=2
        )

    with gr.Row():
        run_btn = gr.Button("Run query")

    with gr.Row():
        sql_box = gr.Textbox(
            label="Generated SQL",
            interactive=False
        )

    with gr.Row():
        result_df = gr.Dataframe(
            label="Query result",
            interactive=False
        )

    with gr.Row():
        plot_out = gr.Plot(label="Visualization")

    error_box = gr.Markdown()

    def on_submit(q):
        sql, df, fig, err = query_agent_with_viz(q)
        if df is None:
            df = pd.DataFrame()
        err_text = f"**Error:** {err}" if err else ""
        return sql, df, fig, err_text

    run_btn.click(
        fn=on_submit,
        inputs=[question_box],
        outputs=[sql_box, result_df, plot_out, error_box]
    )

    gr.Examples(
        examples=example_questions,
        inputs=[question_box],
        outputs=[sql_box, result_df, plot_out, error_box],
        fn=on_submit,
        cache_examples=False
    )

# In Colab, use share=True if you want a public link:
demo.launch()          # or demo.launch(share=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a9c7286ab10ee87afe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
